# Demo implementation of all algorithms

In [1]:
# import necessary libraries
import numpy as np
from simulate_data import simulate_views

## Simulate data

In [2]:
matrices = simulate_views(n=100, num_genes=10, M_C=10, M_Z=5, rank=3, seed=0,rho=0.3)
print(matrices.keys())

dict_keys(['G', 'C', 'Z', 'W_C', 'W_G', 'H_G', 'H_C', 'U_G', 'U_C'])


## Run SCoNE

In [14]:
from algorithms.SCoNE import SCoNE_parallel

factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],matrices['C'],matrices['Z'], rank=3,
    alpha=max(matrices['G'].max(),matrices['C'].max())**2,lambda_H_G=1e-4, lambda_H_C=1e-4, lambda_Gloss=1,   # regularization parameters
    num_init=10,init='nndsvda',G_loss_type='kl_div', C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)

In [15]:
from evaluation.reconstruction_evaluation import best_permutation_similarity

best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.8126865072727106

## Run SCoNE (Fro)

In [22]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],matrices['C'],matrices['Z'], rank=3,
    alpha=max(matrices['G'].max(),matrices['C'].max())**2,lambda_H_G=1e-4, lambda_H_C=1e-4, lambda_Gloss=1,         # regularization parameters
    num_init=10,init='nndsvda',G_loss_type='fro', C_loss_type='fro', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.7329688550156039

## Run CoNE

In [23]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],matrices['C'],matrices['Z'], rank=3,
    alpha=0,lambda_H_G=0, lambda_H_C=0, lambda_Gloss=1,         # regularization parameters
    num_init=10,init='nndsvda',G_loss_type='kl_div', C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.8175604925593781

## Run HNMF

In [24]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],matrices['C'],None, rank=3,
    alpha=0,lambda_H_G=0, lambda_H_C=0, lambda_Gloss=1,         # regularization parameters
    num_init=10,init='nndsvda',G_loss_type='kl_div', C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.7418505948327997

## Run HNMF (res)

In [25]:
from algorithms.SCoNE import proj_nonneg

C_resid = proj_nonneg(matrices['C'] - matrices['Z'] @ np.linalg.lstsq(matrices['Z'], matrices['C'],rcond=None)[0])
G_resid = proj_nonneg(matrices['G'] - matrices['Z'] @ np.linalg.lstsq(matrices['Z'], matrices['G'],rcond=None)[0])

factor_matrices, loss_function = SCoNE_parallel(
    G_resid,C_resid,None, rank=3,
    alpha=0,lambda_H_G=0, lambda_H_C=0, lambda_Gloss=1,         # regularization parameters
    num_init=10,init='nndsvda',G_loss_type='fro', C_loss_type='fro', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.7007339385522641

## Run C-CoNE

In [26]:
factor_matrices, loss_function = SCoNE_parallel(
    None,matrices['C'],matrices['Z'], rank=3,
    num_init=10,init='nndsvda',G_loss_type=None, C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.7105370800772725

## Run G-CoNE

In [27]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],None,matrices['Z'], rank=3,
    num_init=10,init='nndsvda',G_loss_type='kl_div', C_loss_type=None, # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.6588300100170478

## Run C-NMF

In [28]:
factor_matrices, loss_function = SCoNE_parallel(
    None,matrices['C'],None, rank=3,
    num_init=10,init='nndsvda',G_loss_type=None, C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.6666500625689096

## Run G-NMF

In [29]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],None,None, rank=3,
    num_init=10,init='nndsvda',G_loss_type='kl_div', C_loss_type=None, # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.6487308422655252

## Run RGWAS

In [8]:
import os
from algorithms.RGWASWrapper import RGWASWrapper

cwd = os.getcwd()
parent_dir = os.path.dirname(os.getcwd())
# NOTE: important to save as float64 so read correctly in R
np.save(f'{parent_dir}/example_data/G',matrices["G"].astype(np.float64)) 
np.save(f'{parent_dir}/example_data/C',matrices["C"].astype(np.float64))
np.save(f'{parent_dir}/example_data/Z',matrices["Z"][:,1:].astype(np.float64)) # drop one column to avoid multicollinearity

factor_matrices, loss_function = RGWASWrapper(
    r_path='/gpfs/commons/home/anewbury/miniconda/bin/Rscript', # REPLACE WITH CORRECT Rscript path
    G_path=f'{parent_dir}/example_data/G.npy', C_path=f'{parent_dir}/example_data/C.npy', 
    Z_path=f'{parent_dir}/example_data/Z.npy', 
    rank=3, num_init=10)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.3650321658121851

## Run MVBC

In [ ]:
import algorithms.MVBCWrapper as MVBCWrapper
import importlib
importlib.reload(MVBCWrapper)

factor_matrices, loss_function = MVBCWrapper.MVBCWrapper(
    G_path=f'{parent_dir}/example_data/G.npy', C_path=f'{parent_dir}/example_data/C.npy', 
    rank=3, lambda_W=1, lambda_H_G=1, lambda_H_C=1,  
    r_path='/gpfs/commons/home/anewbury/miniconda/bin/Rscript') # REPLACE WITH CORRECT Rscript path
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.41754442167683925